In [3]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path.cwd() / 'data'
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / 'data'


In [4]:
network_df = pd.read_csv(DATA_DIR / 'collegenetwork_fulldata.csv')
college_geo_df = pd.read_csv(DATA_DIR / 'College_geo.csv')
private_geo_df = pd.read_csv(DATA_DIR / 'Private_Schools_geo.csv')


In [5]:
college_geo_df.drop(columns=['col_pop', 'col_enroll'], inplace=True)
college_geo_df.rename(columns={
    'college_id ': 'college_id',
    'cname_2 ': 'col_name'
}, inplace=True)

private_geo_df.drop(columns=['hs_pop', 'hs_enroll'], inplace=True)


In [6]:
college_merged = network_df.merge(
    college_geo_df,
    on='college_id',
    how='left'
)

priv_col_merged = college_merged.merge(
    private_geo_df,
    left_on='ppin',
    right_on='hs_id',
    how='left'
)

priv_col_merged.drop(columns=['hs_id_y'], inplace=True, errors='ignore')
priv_col_merged.rename(columns={'hs_id_x': 'hs_id'}, inplace=True)
priv_col_merged.head()


,college_id,hs_id,ncessch,ppin,cycle,hs_type,col_name,col_city,col_st,col_zip,...,col_shplength,hs_city,hs_state,hs_zip,hs_ctyname,cty_fips,hs_lat,hs_long,hs_x,hs_y
0,100654,551266000000,5.512660e+11,NaN,2022,1,ALABAMA A & M UNIVERSITY,NORMAL,AL,35762.0,...,9383.14932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100654,551266000000,5.512660e+11,NaN,2022,1,ALABAMA A & M UNIVERSITY,NORMAL,AL,35762.0,...,9383.14932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100654,510264000000,5.102640e+11,NaN,2023,1,ALABAMA A & M UNIVERSITY,NORMAL,AL,35762.0,...,9383.14932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100654,484563000000,4.845630e+11,NaN,2019,1,ALABAMA A & M UNIVERSITY,NORMAL,AL,35762.0,...,9383.14932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100654,481707000000,4.817070e+11,NaN,2023,1,ALABAMA A & M UNIVERSITY,NORMAL,AL,35762.0,...,9383.14932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
import requests

url = "https://educationdata.urban.org/api/v1/schools/ccd/directory/2023/"

all_rows = []
while url:
    resp = requests.get(url)
    resp.raise_for_status()
    data = resp.json()

    all_rows.extend(data["results"])
    url = data["next"]   # will be None when there are no more pages

pub_df = pd.DataFrame(all_rows)

pub_df.shape
pub_df.head()


,year,ncessch,school_id,school_name,leaid,lea_name,state_leaid,seasch,street_mailing,city_mailing,...,high_cedp,middle_cedp,ungrade_cedp,enrollment,state_leg_district_lower,state_leg_district_upper,ncessch_num,congress_district_id,direct_certification,lunch_program
0,2023,010000500870,0100870,Albertville Middle School,0100005,Albertville City,AL-101,101-0010,600 E Alabama Ave,Albertville,...,0,1,0,884.0,01026,01009,10000500870,104,654.0,2.0
1,2023,010000500871,0100871,Albertville High School,0100005,Albertville City,AL-101,101-0020,402 E McCord Ave,Albertville,...,1,0,0,1710.0,01026,01009,10000500871,104,1225.0,2.0
2,2023,010000500879,0100879,Albertville Intermediate School,0100005,Albertville City,AL-101,101-0110,901 W McKinney Ave,Albertville,...,0,0,0,838.0,01026,01009,10000500879,104,640.0,2.0
3,2023,010000500889,0100889,Albertville Elementary School,0100005,Albertville City,AL-101,101-0200,145 West End Drive,Albertville,...,0,0,0,929.0,01026,01009,10000500889,104,708.0,2.0
4,2023,010000501616,0101616,Albertville Kindergarten and PreK,0100005,Albertville City,AL-101,101-0035,257 Country Club Rd,Albertville,...,0,0,0,598.0,01026,01009,10000501616,104,330.0,2.0


In [9]:
print(pub_df.columns)

Index(['year', 'ncessch', 'school_id', 'school_name', 'leaid', 'lea_name',
       'state_leaid', 'seasch', 'street_mailing', 'city_mailing',
       'state_mailing', 'zip_mailing', 'street_location', 'city_location',
       'state_location', 'zip_location', 'phone', 'fips', 'latitude',
       'longitude', 'csa', 'cbsa', 'urban_centric_locale', 'county_code',
       'school_level', 'school_type', 'school_status', 'lowest_grade_offered',
       'highest_grade_offered', 'bureau_indian_education', 'title_i_status',
       'title_i_eligible', 'title_i_schoolwide', 'charter', 'magnet',
       'shared_time', 'virtual', 'teachers_fte', 'free_lunch',
       'reduced_price_lunch', 'free_or_reduced_price_lunch', 'elem_cedp',
       'high_cedp', 'middle_cedp', 'ungrade_cedp', 'enrollment',
       'state_leg_district_lower', 'state_leg_district_upper', 'ncessch_num',
       'congress_district_id', 'direct_certification', 'lunch_program'],
      dtype='object')


In [11]:
cols_to_keep = ["ncessch", "city_location", "state_location", "zip_location", "county_code", "latitude", "longitude", ]
pub_geo = pub_df[cols_to_keep].copy()

pub_geo.head()

,ncessch,city_location,state_location,zip_location,county_code,latitude,longitude
0,010000500870,Albertville,AL,35950,1095,34.2602,-86.206200
1,010000500871,Albertville,AL,35950,1095,34.2622,-86.204900
2,010000500879,Albertville,AL,35950,1095,34.2733,-86.220100
3,010000500889,Albertville,AL,35950,1095,34.2527,-86.221806
4,010000501616,Albertville,AL,35951,1095,34.2898,-86.193300


In [14]:
priv_col_merged['ncessch'] = priv_col_merged['ncessch'].astype(str)
pub_geo['ncessch'] = pub_geo['ncessch'].astype(str)

priv_col_merged = priv_col_merged.merge(
    pub_geo,
    on='ncessch',
    how='left',
    suffixes=('', '_pub')
)

# Fill in empty hs columns with public school data
priv_col_merged['hs_city'] = priv_col_merged['hs_city'].fillna(priv_col_merged['city_location'])
priv_col_merged['hs_state'] = priv_col_merged['hs_state'].fillna(priv_col_merged['state_location'])
priv_col_merged['hs_zip'] = priv_col_merged['hs_zip'].fillna(priv_col_merged['zip_location'])
priv_col_merged['cty_fips'] = priv_col_merged['cty_fips'].fillna(priv_col_merged['county_code'])
priv_col_merged['hs_lat'] = priv_col_merged['hs_lat'].fillna(priv_col_merged['latitude'])
priv_col_merged['hs_long'] = priv_col_merged['hs_long'].fillna(priv_col_merged['longitude'])

# Drop the extra pub columns
priv_col_merged.drop(columns=['city_location', 'state_location', 'zip_location', 'latitude', 'longitude', 'county_code'], inplace=True)

priv_col_merged.head()

/var/folders/wz/pjx159kj141cgdkvkdrlfb040000gn/T/ipykernel_97913/1090414385.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  priv_col_merged['hs_zip'] = priv_col_merged['hs_zip'].fillna(priv_col_merged['zip_location'])
/var/folders/wz/pjx159kj141cgdkvkdrlfb040000gn/T/ipykernel_97913/1090414385.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  priv_col_merged['cty_fips'] = priv_col_merged['cty_fips'].fillna(priv_col_merged['county_code'])


,college_id,hs_id,ncessch,ppin,cycle,hs_type,col_name,col_city,col_st,col_zip,...,col_shplength,hs_city,hs_state,hs_zip,hs_ctyname,cty_fips,hs_lat,hs_long,hs_x,hs_y
0,100654,551266000000,551266000000.0,NaN,2022,1,ALABAMA A & M UNIVERSITY,NORMAL,AL,35762.0,...,9383.14932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100654,551266000000,551266000000.0,NaN,2022,1,ALABAMA A & M UNIVERSITY,NORMAL,AL,35762.0,...,9383.14932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100654,510264000000,510264000000.0,NaN,2023,1,ALABAMA A & M UNIVERSITY,NORMAL,AL,35762.0,...,9383.14932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100654,484563000000,484563000000.0,NaN,2019,1,ALABAMA A & M UNIVERSITY,NORMAL,AL,35762.0,...,9383.14932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100654,481707000000,481707000000.0,NaN,2023,1,ALABAMA A & M UNIVERSITY,NORMAL,AL,35762.0,...,9383.14932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
priv_col_merged.drop(columns=["hs_x", "hs_y"], inplace=True)
print(priv_col_merged.columns)

Index(['college_id', 'hs_id', 'ncessch', 'ppin', 'cycle', 'hs_type',
       'col_name', 'col_city', 'col_st', 'col_zip', 'col_type', 'col_ctyname',
       'col_ctyfips', 'col_shparea', 'col_shplength', 'hs_city', 'hs_state',
       'hs_zip', 'hs_ctyname', 'cty_fips', 'hs_lat', 'hs_long'],
      dtype='object')


In [21]:
# Convert columns with NaN to nullable integer type
int_cols = ['col_zip', 'col_type', 'cty_fips']
for col in int_cols:
    priv_col_merged[col] = priv_col_merged[col].astype('Int64')

# Fix ncessch - convert from string with .0 to Int64
priv_col_merged['ncessch'] = priv_col_merged['ncessch'].str.replace('.0', '', regex=False)
priv_col_merged['ncessch'] = pd.to_numeric(priv_col_merged['ncessch'], errors='coerce').astype('Int64')

priv_col_merged.to_csv(DATA_DIR / 'merged_geo.csv', index=False)

AttributeError: Can only use .str accessor with string values!

In [22]:
# Pull college directory data (inst_control, inst_size) - single year is fine for these
url = "https://educationdata.urban.org/api/v1/college-university/ipeds/directory/2022/"
all_rows = []
while url:
    resp = requests.get(url)
    resp.raise_for_status()
    data = resp.json()
    all_rows.extend(data["results"])
    url = data["next"]

college_dir_df = pd.DataFrame(all_rows)
college_char = college_dir_df[['unitid', 'inst_control', 'inst_size']].copy()
college_char.rename(columns={'unitid': 'college_id'}, inplace=True)

print("college_char shape:", college_char.shape)
college_char.head()

college_char shape: (6256, 3)


,college_id,inst_control,inst_size
0,100654,1,3
1,100663,1,5
2,100690,2,1
3,100706,1,3
4,100724,1,2


In [23]:
# Pull admissions data for years 2019-2023
years = [2019, 2020, 2021, 2022, 2023]
all_admissions = []

for year in years:
    url = f"https://educationdata.urban.org/api/v1/college-university/ipeds/admissions-enrollment/{year}/"
    while url:
        resp = requests.get(url)
        resp.raise_for_status()
        data = resp.json()
        all_admissions.extend(data["results"])
        url = data["next"]
    print(f"Finished year {year}")

admissions_df = pd.DataFrame(all_admissions)
college_admissions = admissions_df[['unitid', 'year', 'number_applied', 'number_admitted', 'number_enrolled_total']].copy()
college_admissions.rename(columns={'unitid': 'college_id', 'year': 'cycle'}, inplace=True)

print("college_admissions shape:", college_admissions.shape)
college_admissions.head()

Finished year 2019
Finished year 2020
Finished year 2021
Finished year 2022
Finished year 2023
college_admissions shape: (26629, 5)


,college_id,cycle,number_applied,number_admitted,number_enrolled_total
0,100654,2019,3160.0,2828.0,652.0
1,100654,2019,6419.0,5961.0,1058.0
2,100654,2019,9579.0,8789.0,1710.0
3,100663,2019,2957.0,2279.0,909.0
4,100663,2019,5341.0,3833.0,1437.0


In [ ]:
# Calculate acceptance rate and enrollment rate
college_admissions['acceptance_rate'] = college_admissions['number_admitted'] / college_admissions['number_applied']
college_admissions['enrollment_rate'] = college_admissions['number_enrolled_total'] / college_admissions['number_admitted']

college_admissions.head()

In [25]:
# Rename columns to have col_ prefix
college_admissions.rename(columns={
    'number_applied': 'col_number_applied',
    'number_admitted': 'col_number_admitted',
    'number_enrolled_total': 'col_number_enrolled_total',
    'acceptance_rate': 'col_acceptance_rate',
    'enrollment_rate': 'col_enrollment_rate'
}, inplace=True)

college_char.rename(columns={
    'inst_control': 'col_inst_control',
    'inst_size': 'col_inst_size'
}, inplace=True)

print(college_admissions.columns.tolist())
print(college_char.columns.tolist())

['college_id', 'cycle', 'col_number_applied', 'col_number_admitted', 'col_number_enrolled_total', 'col_acceptance_rate', 'col_enrollment_rate']
['college_id', 'col_inst_control', 'col_inst_size']


In [26]:
# Merge college characteristics (on college_id only)
merged_df = priv_col_merged.merge(college_char, on='college_id', how='left')

# Merge admissions data (on college_id and cycle)
merged_df = merged_df.merge(college_admissions, on=['college_id', 'cycle'], how='left')

print("merged_df shape:", merged_df.shape)
merged_df.head()

merged_df shape: (2618507, 29)


,college_id,hs_id,ncessch,ppin,cycle,hs_type,col_name,col_city,col_st,col_zip,...,cty_fips,hs_lat,hs_long,col_inst_control,col_inst_size,col_number_applied,col_number_admitted,col_number_enrolled_total,col_acceptance_rate,col_enrollment_rate
0,100654,551266000000,551266000000,NaN,2022,1,ALABAMA A & M UNIVERSITY,NORMAL,AL,35762,...,<NA>,NaN,NaN,1.0,3.0,2924.0,1932.0,689.0,0.660739,0.356625
1,100654,551266000000,551266000000,NaN,2022,1,ALABAMA A & M UNIVERSITY,NORMAL,AL,35762,...,<NA>,NaN,NaN,1.0,3.0,5983.0,4160.0,1035.0,0.695303,0.248798
2,100654,551266000000,551266000000,NaN,2022,1,ALABAMA A & M UNIVERSITY,NORMAL,AL,35762,...,<NA>,NaN,NaN,1.0,3.0,0.0,0.0,0.0,NaN,NaN
3,100654,551266000000,551266000000,NaN,2022,1,ALABAMA A & M UNIVERSITY,NORMAL,AL,35762,...,<NA>,NaN,NaN,1.0,3.0,8907.0,6092.0,1724.0,0.683956,0.282994
4,100654,551266000000,551266000000,NaN,2022,1,ALABAMA A & M UNIVERSITY,NORMAL,AL,35762,...,<NA>,NaN,NaN,1.0,3.0,2924.0,1932.0,689.0,0.660739,0.356625


In [31]:
# Pull school-level enrollment by race (2019-2023)
# Using grade-99 (total) and filtering by each race code
import time

years = [2019, 2020, 2021, 2022, 2023]
race_codes = [1, 2, 3, 4, 5, 6, 7]  # White, Black, Hispanic, Asian, AIAN, NHPI, Two or more
all_school_enroll = []

for year in years:
    for race in race_codes:
        url = f"https://educationdata.urban.org/api/v1/schools/ccd/enrollment/{year}/grade-99/race/?race={race}&sex=99"
        while url:
            try:
                resp = requests.get(url, timeout=120)
                resp.raise_for_status()
                data = resp.json()
                all_school_enroll.extend(data["results"])
                url = data["next"]
            except Exception as e:
                print(f"Error for year {year}, race {race}: {e}")
                url = None
    print(f"Finished year {year}, total rows: {len(all_school_enroll)}")

school_enroll_df = pd.DataFrame(all_school_enroll)
print("school_enroll_df shape:", school_enroll_df.shape)
school_enroll_df.head()

Finished year 2019, total rows: 695030
Finished year 2020, total rows: 1390200
Finished year 2021, total rows: 2090403
Finished year 2022, total rows: 2789423
Finished year 2023, total rows: 3487169
school_enroll_df shape: (3487169, 9)


,year,ncessch,ncessch_num,grade,race,sex,enrollment,fips,leaid
0,2019,010000500870,10000500870,99,1,99,351,1,100005
1,2019,010000500871,10000500871,99,1,99,711,1,100005
2,2019,010000500879,10000500879,99,1,99,378,1,100005
3,2019,010000500889,10000500889,99,1,99,352,1,100005
4,2019,010000501616,10000501616,99,1,99,230,1,100005


In [ ]:
# Pull district-level enrollment by race (2019-2023)
all_district_enroll = []

for year in years:
    for race in race_codes:
        url = f"https://educationdata.urban.org/api/v1/school-districts/ccd/enrollment/{year}/grade-99/race/?race={race}&sex=99"
        while url:
            try:
                resp = requests.get(url, timeout=120)
                resp.raise_for_status()
                data = resp.json()
                all_district_enroll.extend(data["results"])
                url = data["next"]
            except Exception as e:
                print(f"Error for year {year}, race {race}: {e}")
                url = None
    print(f"Finished district year {year}, total rows: {len(all_district_enroll)}")

district_enroll_df = pd.DataFrame(all_district_enroll)
print("district_enroll_df shape:", district_enroll_df.shape)
district_enroll_df.head()

In [ ]:
# Pivot school enrollment data to get race percentages
school_pivot = school_enroll_df.pivot_table(
    index=['ncessch', 'year'],
    columns='race',
    values='enrollment',
    aggfunc='sum'
).reset_index()

# Calculate total enrollment and percentages
# Race codes: 1=White, 2=Black, 3=Hispanic, 4=Asian, 5=AIAN, 6=NHPI, 7=Two or more, 9=Unknown, 99=Total
race_cols = [c for c in [1, 2, 3, 4, 5, 6, 7] if c in school_pivot.columns]
school_pivot['total'] = school_pivot[race_cols].sum(axis=1, skipna=True)

school_pivot['hs_pct_white'] = school_pivot.get(1, 0) / school_pivot['total']
school_pivot['hs_pct_black'] = school_pivot.get(2, 0) / school_pivot['total']
school_pivot['hs_pct_hispanic'] = school_pivot.get(3, 0) / school_pivot['total']
school_pivot['hs_pct_asian'] = school_pivot.get(4, 0) / school_pivot['total']
school_pivot['hs_pct_aian'] = school_pivot.get(5, 0) / school_pivot['total']
school_pivot['hs_pct_nhpi'] = school_pivot.get(6, 0) / school_pivot['total']
school_pivot['hs_pct_two_or_more'] = school_pivot.get(7, 0) / school_pivot['total']

# Keep only needed columns
school_race = school_pivot[['ncessch', 'year', 'hs_pct_white', 'hs_pct_black', 'hs_pct_hispanic', 
                            'hs_pct_asian', 'hs_pct_aian', 'hs_pct_nhpi', 'hs_pct_two_or_more']].copy()
school_race.rename(columns={'year': 'cycle'}, inplace=True)
school_race['hs_is_district'] = 0

print("school_race shape:", school_race.shape)
school_race.head()

In [ ]:
# Pivot district enrollment data to get race percentages
district_pivot = district_enroll_df.pivot_table(
    index=['leaid', 'year'],
    columns='race',
    values='enrollment',
    aggfunc='sum'
).reset_index()

# Calculate total enrollment and percentages
race_cols = [c for c in [1, 2, 3, 4, 5, 6, 7] if c in district_pivot.columns]
district_pivot['total'] = district_pivot[race_cols].sum(axis=1, skipna=True)

district_pivot['hs_pct_white'] = district_pivot.get(1, 0) / district_pivot['total']
district_pivot['hs_pct_black'] = district_pivot.get(2, 0) / district_pivot['total']
district_pivot['hs_pct_hispanic'] = district_pivot.get(3, 0) / district_pivot['total']
district_pivot['hs_pct_asian'] = district_pivot.get(4, 0) / district_pivot['total']
district_pivot['hs_pct_aian'] = district_pivot.get(5, 0) / district_pivot['total']
district_pivot['hs_pct_nhpi'] = district_pivot.get(6, 0) / district_pivot['total']
district_pivot['hs_pct_two_or_more'] = district_pivot.get(7, 0) / district_pivot['total']

# Keep only needed columns
district_race = district_pivot[['leaid', 'year', 'hs_pct_white', 'hs_pct_black', 'hs_pct_hispanic', 
                                'hs_pct_asian', 'hs_pct_aian', 'hs_pct_nhpi', 'hs_pct_two_or_more']].copy()
district_race.rename(columns={'year': 'cycle'}, inplace=True)
district_race['hs_is_district'] = 1

print("district_race shape:", district_race.shape)
district_race.head()

In [ ]:
# Merge school race data to main dataframe on ncessch and cycle
school_race['ncessch'] = school_race['ncessch'].astype(str)
merged_df['ncessch'] = merged_df['ncessch'].astype(str)

merged_df = merged_df.merge(
    school_race,
    on=['ncessch', 'cycle'],
    how='left'
)

# For rows that didn't match on ncessch (districts), try matching on hs_id as leaid
# Extract leaid from hs_id (first 7 digits for districts)
merged_df['leaid_temp'] = merged_df['hs_id'].astype(str).str[:7]
district_race['leaid'] = district_race['leaid'].astype(str)

# Merge district data for unmatched rows
district_merge = merged_df[merged_df['hs_pct_white'].isna()].drop(
    columns=['hs_pct_white', 'hs_pct_black', 'hs_pct_hispanic', 'hs_pct_asian', 
             'hs_pct_aian', 'hs_pct_nhpi', 'hs_pct_two_or_more', 'hs_is_district']
).merge(
    district_race,
    left_on=['leaid_temp', 'cycle'],
    right_on=['leaid', 'cycle'],
    how='left'
)

# Update main dataframe with district data
merged_df.update(district_merge)

# Clean up temp column
merged_df.drop(columns=['leaid_temp'], inplace=True, errors='ignore')

print("merged_df shape:", merged_df.shape)
print("Non-null hs_pct_white:", merged_df['hs_pct_white'].notna().sum())
merged_df.head()